In [0]:
from pyspark.sql.functions import col, expr, regexp_extract, trim

restaurants_silver_df = spark.table(
    "workspace.zomato_silver.restaurants"
)

dim_restaurants_df = (
    restaurants_silver_df
    .withColumn(
        "restaurant_id",
        expr("try_cast(id AS BIGINT)")
    )
    .withColumn(
        "rating",
        expr("try_cast(rating AS DOUBLE)")
    )
    .withColumn(
        "cost_amount",
        regexp_extract(col("cost"), r"(\d+(?:\.\d+)?)", 1)
        .cast("double")
    )
    .filter(col("restaurant_id").isNotNull())
    .select(
        "restaurant_id",
        "name",
        "city",
        "rating",
        "rating_count",
        "cost_amount",
        "cuisine",
        "lic_no",
        "link",
        "address",
        "menu"
    )
    .dropDuplicates(["restaurant_id"])
)

display(dim_restaurants_df.limit(10))

restaurant_id,name,city,rating,rating_count,cost_amount,cuisine,lic_no,link,address,menu
373319,Nikku Chaap Cafe,Abohar,null,Too Few Ratings,200.0,"Indian,Chinese",license,https://www.swiggy.com/restaurants/nikku-chaap-cafe-central-canal-colony-abohar-373319,"Nikku Chaap Cafe, 324, Street No: 3, Badi Puri, Nai Abadi, Abohar-152116",Menu/373319.json
226299,Lotus Grand Family Restaurant,Adilabad,3.6,50+ ratings,250.0,Biryani,13621001000019,https://www.swiggy.com/restaurants/lotus-grand-family-restaurant-city-ravindra-nagar-colony-adilabad-226299,"Lotus Grand Family Restaurant, opp bustand Adilabad",Menu/226299.json
578207,SAMBHU DOSA,Adityapur,null,Too Few Ratings,150.0,"South Indian,Chinese",21121083000288,https://www.swiggy.com/restaurants/sambhu-dosa-adityapur-jamshedpur-578207,"SAMBHU DOSA, SHOP NO 103 GROUND FLOOR ADITYAPUR DINDLI BASTI, ADITYAPUR, GAMAHARIA, Saraikela, Jharkhand - 831013",Menu/578207.json
102042,Biryani House,Adityapur,2.9,20+ ratings,499.0,"Biryani,Indian",11117006000117,https://www.swiggy.com/restaurants/biryani-house-bistupur-jamshedpur-102042,"Biryani House, Crystal Square, kharkhai road link road, Bistupur, Jamshedpur",Menu/102042.json
154840,Anand,Adityapur,4.3,500+ ratings,299.0,"South Indian,Indian",11120006000027,https://www.swiggy.com/restaurants/anand-bistupur-jamshedpur-154840,"Anand, 7 J-road bistupr",Menu/154840.json
174458,Milanee s Kitchen,Adityapur,4.2,100+ ratings,399.0,"Bengali,Indian",11120006000071,https://www.swiggy.com/restaurants/milanee-s-kitchen-womens-college-bistupur-jamshedpur-174458,"Milanee s Kitchen, J Road, infront of women's college, bistupur, jamshedpur",Menu/174458.json
486482,The Moti Mahal,Adityapur,3.2,100+ ratings,399.0,"Indian,Chinese",21122254000062,https://www.swiggy.com/restaurants/the-moti-mahal-bistupur-jamshedpur-486482,"The Moti Mahal, BESIDE CHAGANLAL JEWELERS OPP FLOWER SHOP BISTUPUR MARKET, Bistupur, Muncipality, Purbi Singhbhum, Jharkhand - 831001",Menu/486482.json
387177,New Gangour Sweets,Adityapur,4.1,50+ ratings,200.0,"Sweets,Snacks",11118006000168,https://www.swiggy.com/restaurants/new-gangour-sweets-bistupur-jamshedpur-387177,"New Gangour Sweets, SBI BUILDING, RAILWAY CROSSING, JUGSALAI, JAMSHEDPUR22",Menu/387177.json
487422,Foodie Baba,Adityapur,null,Too Few Ratings,200.0,"Chinese,South Indian",21120272000100,https://www.swiggy.com/restaurants/foodie-baba-adityapur-jamshedpur-487422,"Foodie Baba, M-43, RIT, Co-operative society, Ichhapur Durgapuja Maidan, Gwalapara, Adityapur-2, Adityapur, Jamshedpur",Menu/487422.json
518819,Kebabchi,Adityapur,null,Too Few Ratings,300.0,"Indian,Snacks",21122254000057,https://www.swiggy.com/restaurants/kebabchi-bistupur-jamshedpur-518819,"Kebabchi, MUNESHWARI BHAWAN, ROAD NO 2, B1, CONTRACTORS AREA, PS BISTUPUR,, Bistupur, Muncipality, Purbi Singhbhum, Jharkhand - 831001",Menu/518819.json


In [0]:
print("Restaurant dimension rows:", dim_restaurants_df.count())
print(
    "Distinct restaurant IDs:",
    dim_restaurants_df.select("restaurant_id").distinct().count()
)

Restaurant dimension rows: 148541
Distinct restaurant IDs: 148541


In [0]:
dim_restaurants_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.zomato_gold.dim_restaurants")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.dim_restaurants
        LIMIT 10
    """)
)

restaurant_id,name,city,rating,rating_count,cost_amount,cuisine,lic_no,link,address,menu
574413,STEAMIN MUGS CAFE & RESTAURANT,Agartala,null,Too Few Ratings,400.0,"North Indian,Chinese",12522010000117,https://www.swiggy.com/restaurants/steamin-mugs-cafe-and-restaurant-gb-road-agartala-574413,"STEAMIN MUGS CAFE & RESTAURANT, KUNJABAN, AGARTALA TRIPURA WEST, Agartala Municiapal Corporation, Agartala Municipal Council, Tripura-799001",Menu/574413.json
497414,Toro Toro,"Vastrapur,Ahmedabad",null,Too Few Ratings,800.0,"Chinese,Indian",20722038000614,https://www.swiggy.com/restaurants/toro-toro-vastrapur-ahmedabad-497414,"Toro Toro, Aurobindo Society Rd, Vastrapur, Ahmedabad, Gujarat, India",Menu/497414.json
460232,Ad Momos And Chinese Foods,"GOTA,Ahmedabad",null,Too Few Ratings,150.0,"Chinese,Pizzas",20721037002677,https://www.swiggy.com/restaurants/ad-momos-and-chinese-foods-gota-ahmedabad-460232,"Ad Momos And Chinese Foods, RAIN FOREST RESTAURANT, Sarkhej - Gandhinagar Hwy, opp. MANAN AUTOLINK, Gota, Ahmedabad, Gujarat 382481, India",Menu/460232.json
486864,Junior & Senior Burger,"Paldi & Ambawadi,Ahmedabad",null,Too Few Ratings,200.0,Fast Food,10719026001382,https://www.swiggy.com/restaurants/junior-and-senior-burger-paldi-and-ambawadi-ahmedabad-486864,"Junior & Senior Burger, GF/07, DEV complex, Parimal Garden char rasta, Ellisbridge, 380006",Menu/486864.json
483683,Gayatri Bhajipav & Cold Drinks,"Ghatlodia,Ahmedabad",null,Too Few Ratings,300.0,Fast Food,20720033000067,https://www.swiggy.com/restaurants/gayatri-bhajipav-and-cold-drinks-ghatlodia-satadhar-cross-road-ahmedabad-483683,"Gayatri Bhajipav & Cold Drinks, G11, Centar Plaza, Opp Sarthak School, Satadhar Cross Road, Sola, Ahmedabad - 380061",Menu/483683.json
357663,Mirch Masala,"Ghatlodia,Ahmedabad",4.0,20+ ratings,200.0,"North Indian,Punjabi",license,https://www.swiggy.com/restaurants/mirch-masala-ghatlodia-thaltej-ahmedabad-357663,"Mirch Masala, 103, City Centre 2, Science City Road, Sola, Ahmedabad",Menu/357663.json
422249,Krishna Snacks,"Bopal,Ahmedabad",null,Too Few Ratings,250.0,"North Indian,Chinese",20721038000768,https://www.swiggy.com/restaurants/krishna-snacks-bopal-ahmedabad-422249,"Krishna Snacks, 3, Poojan Complex, Beside Kabir Complex, Bopal - Ghuma Rd, opposite Homeopathic College, Bopal, Ahmedabad, Gujarat 380058, India",Menu/422249.json
149546,Cakelelo.Com-Live Cake Studio,"Gandhinagar,Ahmedabad",null,Too Few Ratings,500.0,"Bakery,Desserts",20718009000353,https://www.swiggy.com/restaurants/cakelelo-com-live-cake-studio-airport-highway-gandhinagar-ahmedabad-149546,"Cakelelo.Com-Live Cake Studio, A-8, Pramukh Arcade, Reliance Cross Road, Kudasan, Airport Gandhinagar Highway, Gandhinagar",Menu/149546.json
364992,Food Paradise,"Gandhinagar,Ahmedabad",3.7,100+ ratings,300.0,"North Indian,Chinese",20721009000124,https://www.swiggy.com/restaurants/food-paradise-gandhinagar-kudasan-ahmedabad-364992,"Food Paradise, SHOP NO.4 PRAMUKH CYPRUS, Sargasan , Gandhinagar, Gandhinagar, Gujarat - 382421",Menu/364992.json
323008,Chhasswala,"Navrangpura,Ahmedabad",4.4,20+ ratings,150.0,"Street Food,Beverages",license,https://www.swiggy.com/restaurants/chhasswala-navrangpura-ahmedabad-323008,"Chhasswala, GF-3 , HSG House , Nr. Vijay Restaurant , Drive In Road , Navrangpura , Ahmedabad , Gujarat - 380009",Menu/323008.json


In [0]:
from pyspark.sql.functions import col, trim

foods_silver_df = spark.table(
    "workspace.zomato_silver.foods"
)

dim_foods_df = (
    foods_silver_df
    .withColumn("food_id", trim(col("f_id")))
    .withColumn("food_name", trim(col("item")))
    .withColumn("food_type", trim(col("veg_or_non_veg")))
    .filter(col("food_id").isNotNull())
    .select(
        "food_id",
        "food_name",
        "food_type"
    )
    .dropDuplicates(["food_id"])
)

display(dim_foods_df.limit(10))

print("Food dimension rows:", dim_foods_df.count())

print(
    "Distinct food IDs:",
    dim_foods_df.select("food_id").distinct().count()
)

food_id,food_name,food_type
fd4,Maxican Burger,Veg
fd9,Pink Sauce,Veg
fd17,Paneer Noodles,Veg
fd24,Spring Roll [2pc],Veg
fd29,Chicken Fried Rice,Non-veg
fd39,Classical Fries,Veg
fd72,Creamy Pav Bhaji,Veg
fd82,Capcicum Onion Pizza,Veg
fd94,Mango Shake,Veg
fd130,Roohafza Milkshake,Veg


Food dimension rows: 371563
Distinct food IDs: 371563


In [0]:
dim_foods_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.zomato_gold.dim_foods")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.dim_foods
        LIMIT 10
    """)
)

food_id,food_name,food_type
fd4,Maxican Burger,Veg
fd9,Pink Sauce,Veg
fd17,Paneer Noodles,Veg
fd24,Spring Roll [2pc],Veg
fd29,Chicken Fried Rice,Non-veg
fd39,Classical Fries,Veg
fd72,Creamy Pav Bhaji,Veg
fd82,Capcicum Onion Pizza,Veg
fd94,Mango Shake,Veg
fd130,Roohafza Milkshake,Veg


In [0]:
users_silver_df = spark.table(
    "workspace.zomato_silver.users"
)

dim_users_df = (
    users_silver_df
    .withColumn("user_id", col("user_id").cast("long"))
    .withColumn("name", trim(col("name")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("marital_status", trim(col("marital_status")))
    .withColumn("occupation", trim(col("occupation")))
    .withColumn("monthly_income", trim(col("monthly_income")))
    .withColumn(
        "educational_qualifications",
        trim(col("educational_qualifications"))
    )
    .filter(col("user_id").isNotNull())
    .select(
        "user_id",
        "name",
        "email",
        "age",
        "gender",
        "marital_status",
        "occupation",
        "monthly_income",
        "educational_qualifications",
        "family_size"
    )
    .dropDuplicates(["user_id"])
)

display(dim_users_df.limit(10))

print("User dimension rows:", dim_users_df.count())

print(
    "Distinct user IDs:",
    dim_users_df.select("user_id").distinct().count()
)

user_id,name,email,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size
12,Michael Gilbert,wendycollins@example.com,23,Male,Single,Student,Below Rs.10000,Post Graduate,2
18,Antonio Brown,meyernicole@example.net,23,Female,Single,Student,No Income,Graduate,3
38,Brandon Blankenship,kylestevens@example.org,32,Female,Prefer not to say,House wife,No Income,Graduate,5
67,Erin Torres,kevin73@example.org,24,Male,Single,Employee,10001 to 25000,Graduate,4
70,Sara Davis,belinda79@example.com,24,Female,Married,Employee,More than 50000,Ph.D,4
93,Gabriella Johnson,melissa97@example.org,22,Male,Single,Student,No Income,Post Graduate,2
161,Lawrence Khan,shanehernandez@example.net,21,Male,Single,Student,No Income,Graduate,2
186,Joseph Stevens,scott97@example.com,28,Male,Married,Employee,More than 50000,Post Graduate,1
190,Dr. Nathaniel Sanchez,careykyle@example.org,31,Male,Married,Self Employeed,More than 50000,School,6
218,William Wheeler,megan32@example.org,26,Male,Single,Employee,10001 to 25000,Graduate,2


User dimension rows: 100000
Distinct user IDs: 100000


In [0]:
dim_users_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.zomato_gold.dim_users")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.zomato_gold.dim_users
        LIMIT 10
    """)
)

user_id,name,email,age,gender,marital_status,occupation,monthly_income,educational_qualifications,family_size
93,Gabriella Johnson,melissa97@example.org,22,Male,Single,Student,No Income,Post Graduate,2
295,Andrew Patrick,johnkennedy@example.org,25,Female,Prefer not to say,Employee,25001 to 50000,Post Graduate,3
300,Diana Murphy,patrick47@example.com,25,Female,Single,Employee,More than 50000,Post Graduate,6
585,Desiree Reeves,berryjoseph@example.net,19,Male,Single,Student,No Income,Graduate,6
633,Yvonne Smith,cgalvan@example.net,30,Male,Married,Self Employeed,More than 50000,Graduate,1
849,Angela Carlson,jacqueline47@example.net,25,Female,Single,Student,No Income,Post Graduate,3
1044,Tanner Wilson,nortiz@example.com,25,Female,Single,Employee,25001 to 50000,Post Graduate,2
1238,Ronald Simmons,iwright@example.org,23,Male,Single,Student,No Income,Post Graduate,2
1279,Jerry Blair,mmccullough@example.org,23,Male,Single,Employee,10001 to 25000,Post Graduate,2
1283,Gregory Lloyd,courtneysmith@example.com,32,Female,Married,Employee,25001 to 50000,Graduate,5
